# AI 面试手写题库（精简实现）

作者: ChatGPT（教学版实现）

说明: 本 notebook 汇总了常见面试题的简洁手写实现（PyTorch / NumPy），以便练习面试时的代码书写与原理讲解。很多实现为教学版，未包含工程级优化（CUDA kernels、混合精度、并行等）。

你可以按单元运行每段代码，文件较长，建议按需求逐段练习。

In [ ]:
# 环境准备：导入常用依赖
import math, random, time
from typing import Optional, Tuple, List
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print('依赖导入完成，PyTorch 版本:', torch.__version__)

## Section 1: 大模型注意力 / 编码模块

包含：MHA、FlashAttention 思路示例、GQA/MQA、RoPE/ALiBi/Sinusoidal/T5 相对位置、Masked Attention、Linear Attention、Multi-Scale Attention。

In [ ]:
# 1.1 简易 Multi-Head Attention（可选 causal mask）
class SimpleMHA(nn.Module):
    """简洁版 Multi-Head Attention，带可选 causal mask。教学用，未做工程优化。"""
    def __init__(self, dim, num_heads, dropout=0.0):
        super().__init__()
        assert dim % num_heads == 0
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv = nn.Linear(dim, dim * 3, bias=False)
        self.out = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask: Optional[torch.Tensor]=None, causal=False):
        # x: (B, T, D)
        B, T, D = x.shape
        qkv = self.qkv(x)  # (B, T, 3D)
        qkv = qkv.reshape(B, T, 3, self.num_heads, self.head_dim).permute(2,0,3,1,4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # each: (B, H, T, head_dim)
        # scaled dot-product
        scores = torch.einsum("bhqd,bhkd->bhqk", q, k) / math.sqrt(self.head_dim)  # (B,H,T,T)
        if causal:
            causal_mask = torch.triu(torch.full((T,T), float('-inf'), device=x.device), diagonal=1)
            scores = scores + causal_mask.unsqueeze(0).unsqueeze(0)
        if mask is not None:
            # mask: (B, T) or (B, 1, 1, T) where True indicates pad -> set -inf
            scores = scores + (mask.unsqueeze(1).unsqueeze(1) * -1e9)
        attn = torch.softmax(scores, dim=-1)
        out = torch.einsum("bhqk,bhkd->bhqd", attn, v)  # (B,H,T,hd)
        out = out.permute(0,2,1,3).reshape(B,T,D)
        return self.out(self.dropout(out))

# 测试 MHA
B, T, D, H = 2, 16, 64, 8
x = torch.rand(B, T, D)
mha = SimpleMHA(D, H)
y = mha(x, causal=True)
print('MHA 输出形状:', y.shape)

#############################################################################
# 1.2 FlashAttention 思路示例（块化 attention，教学性）
def flash_attention_blockwise(q, k, v, block_k=256, causal=False):
    """
    教学版 block-wise attention：把 K,V 分块以降低单次内存开销。
    说明：真正的 FlashAttention 使用前缀归一化(log-sum-exp trick)来精确合并 softmax 分块结果。
    这里为展示思路，使用简化的数值处理。
    q: (B, Lq, D)
    k,v: (B, Lk, D)
    """
    B, Lq, D = q.shape
    _, Lk, _ = k.shape
    out = torch.zeros(B, Lq, D, device=q.device, dtype=q.dtype)
    # 对 K 分块
    for k_start in range(0, Lk, block_k):
        k_end = min(Lk, k_start + block_k)
        k_blk = k[:, k_start:k_end, :]  # (B, blk, D)
        v_blk = v[:, k_start:k_end, :]  # (B, blk, D)
        scores = torch.einsum("bqd,bkd->bqk", q, k_blk) / math.sqrt(D)  # (B, Lq, blk)
        if causal:
            # mask out future positions where k index > q index
            k_idx = torch.arange(k_start, k_end, device=q.device).unsqueeze(0)  # (1, blk)
            q_idx = torch.arange(Lq, device=q.device).unsqueeze(1)  # (Lq,1)
            mask = (k_idx.unsqueeze(1) > q_idx.unsqueeze(-1)).unsqueeze(0)  # (1,Lq,blk)
            scores = scores.masked_fill(mask, float("-inf"))
        # stable softmax per block
        max_s, _ = scores.max(dim=-1, keepdim=True)
        exp_s = torch.exp(scores - max_s)
        num = torch.einsum("bqk,bkd->bqd", exp_s, v_blk)
        den = exp_s.sum(dim=-1, keepdim=True)  # (B, Lq, 1)
        out = out + num / (den + 1e-9) * 1.0  # 近似累加
    return out

# 简要示例
B, Lq, Lk, D = 1, 64, 64, 32
q = torch.rand(B, Lq, D)
k = torch.rand(B, Lk, D)
v = torch.rand(B, Lk, D)
out = flash_attention_blockwise(q, k, v, block_k=16)
print('flash_attention_blockwise 输出形状:', out.shape)

#############################################################################
# 1.3 Grouped Query Attention (GQA) / Multi-Query Attention (MQA) 思路演示
class GQA(nn.Module):
    """
    Grouped Query Attention: 将 query head 分为 groups 组，每组共享一对 k/v 投影。
    MQA 是 GQA 的特殊情况（groups == num_heads 或 groups == 1，取决定义）。
    此处仅演示模块化思路，未实现完全的张量重排细节。
    """
    def __init__(self, dim, num_heads, groups=1):
        super().__init__()
        assert num_heads % groups == 0
        self.dim, self.num_heads, self.groups = dim, num_heads, groups
        self.head_dim = dim // num_heads
        self.q_proj = nn.Linear(dim, dim)
        # 为每组生成一对 k/v（示意）
        kv_dim = (num_heads // groups) * self.head_dim * 2 * groups
        self.kv_proj = nn.Linear(dim, kv_dim)
        self.out = nn.Linear(dim, dim)

    def forward(self, x):
        B, T, D = x.shape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).permute(0,2,1,3)
        # kv_proj 输出处理在工业实现中会更规整，这里只标注思想
        kv = self.kv_proj(x)
        # 省略完整展开逻辑：关键点是 group 内共享 kv，节省 KV 参数与内存
        return torch.zeros_like(x)

print('GQA 类定义完成（示意）')

#############################################################################
# 1.4 位置编码: Sinusoidal / RoPE / ALiBi / T5 bucket 示意
def sinusoidal_positional_encoding(dim, seq_len, device='cpu'):
    pos = torch.arange(seq_len, device=device).unsqueeze(1)
    i = torch.arange(dim, device=device).unsqueeze(0)
    angle_rates = pos / (10000 ** (2*(i//2)/dim))
    pe = torch.zeros(seq_len, dim, device=device)
    pe[:, 0::2] = torch.sin(angle_rates[:, 0::2])
    pe[:, 1::2] = torch.cos(angle_rates[:, 1::2])
    return pe  # (seq_len, dim)

def apply_rope(q, k, seq_len, base=10000):
    # q,k: (..., seq_len, head_dim) 假设最后维度为偶数
    device = q.device
    head_dim = q.shape[-1]
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    pos = torch.arange(seq_len, device=device).type_as(q).unsqueeze(1)
    sinusoid_inp = torch.einsum("p,d->pd", pos, inv_freq)  # (seq_len, head_dim/2)
    sin = sinusoid_inp.sin().repeat_interleave(2, dim=1)  # (seq_len, head_dim)
    cos = sinusoid_inp.cos().repeat_interleave(2, dim=1)
    def rotate(x):
        x1 = x[..., ::2]
        x2 = x[..., 1::2]
        rx = torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
        return rx.flatten(-2)
    q_rot = rotate(q)
    k_rot = rotate(k)
    return q_rot, k_rot

def alibi_bias(num_heads, seq_len, device='cpu'):
    slopes = torch.tensor([1.0/(2**(i/num_heads)) for i in range(num_heads)], device=device)
    pos = torch.arange(seq_len, device=device)
    i = pos.view(1, -1, 1)
    j = pos.view(1, 1, -1)
    bias = -(j - i).float() * slopes.view(num_heads, 1, 1)
    return bias  # (H, L, L)

def relative_position_bucket(rel_pos, num_buckets=32, max_distance=128):
    # rel_pos: tensor of distances (j-i)
    n = -rel_pos
    num_buckets_half = num_buckets // 2
    is_small = n < num_buckets_half
    val = torch.where(
        is_small,
        n,
        (num_buckets_half + (torch.log(n.float() / num_buckets_half) / math.log(max_distance / num_buckets_half) * (num_buckets - num_buckets_half)).long())
    )
    val = torch.clamp(val, 0, num_buckets-1)
    return val

#############################################################################
# 1.5 Masked Attention 辅助函数
def create_look_ahead_mask(size):
    return torch.triu(torch.ones(size, size), diagonal=1).bool()

def padding_mask_from_lengths(lengths, max_len=None):
    B = lengths.shape[0]
    if max_len is None:
        max_len = lengths.max().item()
    idx = torch.arange(max_len, device=lengths.device).unsqueeze(0)
    mask = idx >= lengths.unsqueeze(1)  # True for pad positions
    return mask  # (B, max_len)

#############################################################################
# 1.6 Linear Attention（核化注意力）示意
def elu_feature_map(x):
    return F.elu(x) + 1.0

def linear_attention(q, k, v, feature_map=elu_feature_map, eps=1e-6):
    # q,k: (B, T, D), v: (B, T, Dv)
    qk = feature_map(q)  # (B,T,D)
    kk = feature_map(k)
    # 利用结合律： (φ(Q) (φ(K)^T V))
    kv = torch.einsum("btd,bte->bde", kk, v)  # (B, D, Dv)
    denom = torch.einsum("btd,bd->bt", qk, kk.sum(dim=1))  # (B, T)
    out = torch.einsum("btd,bde->bte", qk, kv)
    out = out / (denom.unsqueeze(-1) + eps)
    return out

#############################################################################
# 1.7 Multi-Scale Attention 思路示意（视觉）
def multi_scale_attention(q, kv_list):
    """
    q: (B, Lq, D)
    kv_list: list of (k, v) for different scales
    """
    outs = []
    for k, v in kv_list:
        scores = torch.einsum("bqd,bkd->bqk", q, k) / math.sqrt(q.shape[-1])
        attn = torch.softmax(scores, dim=-1)
        outs.append(torch.einsum("bqk,bkd->bqd", attn, v))
    out = torch.cat(outs, dim=-1)
    return out

print('Section 1 单元定义完成')

## Section 2: 深度学习基础：优化器 / 损失 / 嵌入 / 归一化

包含：SGD+Momentum/Nesterov、AdamW、LARS、RMSprop/Adagrad、LR调度、Label Smoothing、Focal Loss、Smooth L1、Triplet、Dice、KL、BCEWithLogits、LayerNorm/BatchNorm/Embedding/PatchEmbedding 等。

In [ ]:
# 2.1 SGD + Momentum / Nesterov 手写（教学版）
class SGDManual:
    def __init__(self, params, lr=1e-3, momentum=0.0, nesterov=False):
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum
        self.nesterov = nesterov
        self.state = {}
    def step(self, grads):
        # grads: 与 params 对应的梯度列表（教学简化接口）
        for p, g in zip(self.params, grads):
            if p not in self.state:
                self.state[p] = torch.zeros_like(g)
            buf = self.state[p]
            buf.mul_(self.momentum).add_(g)  # v = mu * v + g
            if self.nesterov:
                update = g + self.momentum * buf
            else:
                update = buf
            p.data.add_(-self.lr * update)

# 2.2 AdamW 手写（权重衰减解耦）
class AdamWManual:
    def __init__(self, params, lr=1e-3, betas=(0.9,0.999), eps=1e-8, weight_decay=0.01):
        self.params = list(params); self.lr=lr; self.betas=betas; self.eps=eps; self.wd=weight_decay
        self.state = {}
        self.step_count = 0
    def step(self, grads):
        self.step_count += 1
        beta1, beta2 = self.betas
        for p, g in zip(self.params, grads):
            if p not in self.state:
                self.state[p] = {"m": torch.zeros_like(g), "v": torch.zeros_like(g)}
            s = self.state[p]
            s["m"].mul_(beta1).add_( (1-beta1) * g )
            s["v"].mul_(beta2).add_( (1-beta2) * (g*g) )
            m_hat = s["m"] / (1 - beta1**self.step_count)
            v_hat = s["v"] / (1 - beta2**self.step_count)
            update = m_hat / (v_hat.sqrt() + self.eps)
            # decoupled weight decay
            p.data.add_(-self.lr * (update + self.wd * p.data))

# 2.3 LARS 简洁函数
def lars_update(param, grad, lr, trust_coefficient=0.001, eps=1e-9):
    w_norm = torch.norm(param)
    g_norm = torch.norm(grad)
    local_lr = trust_coefficient * (w_norm / (g_norm + eps))
    step = local_lr * grad
    param.data.add_(-lr * step)

# 2.4 RMSprop / Adagrad（教学版）
class RMSpropManual:
    def __init__(self, params, lr=1e-3, alpha=0.99, eps=1e-8):
        self.params=list(params); self.lr=lr; self.alpha=alpha; self.eps=eps
        self.state={}
    def step(self, grads):
        for p,g in zip(self.params,grads):
            if p not in self.state: self.state[p]=torch.zeros_like(g)
            s=self.state[p]; s.mul_(self.alpha).add_( (1-self.alpha)*g*g )
            p.data.add_(-self.lr * g / (s.sqrt()+self.eps))

class AdagradManual:
    def __init__(self, params, lr=1e-2, eps=1e-10):
        self.params=list(params); self.lr=lr; self.eps=eps; self.state={}
    def step(self, grads):
        for p,g in zip(self.params,grads):
            if p not in self.state: self.state[p]=torch.zeros_like(g)
            s = self.state[p]; s.add_(g*g)
            p.data.add_(-self.lr * g / (s.sqrt()+self.eps))

# 2.5 学习率调度器（StepLR / Warmup+Cosine 例子）
class StepLR:
    def __init__(self, optimizer, step_size, gamma=0.1):
        self.opt = optimizer; self.step_size=step_size; self.gamma=gamma; self.epoch=0
    def step(self):
        self.epoch += 1
        if self.epoch % self.step_size == 0:
            for p in self.opt.param_groups:
                p["lr"] *= self.gamma

class WarmupCosineLR:
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=0.0):
        self.opt=optimizer; self.warmup=warmup_steps; self.total=total_steps; self.min_lr=min_lr; self.step_num=0
    def step(self):
        self.step_num += 1
        if self.step_num < self.warmup:
            factor = self.step_num / max(1, self.warmup)
        else:
            progress = (self.step_num - self.warmup) / max(1, self.total - self.warmup)
            factor = 0.5*(1.0 + math.cos(math.pi * progress))
        for g in self.opt.param_groups:
            g['lr'] = max(self.min_lr, g.get('initial_lr', g['lr']) * factor)

# 2.6 损失函数实现
def label_smoothing_ce(logits, target, smoothing=0.1):
    n_class = logits.size(-1)
    log_probs = F.log_softmax(logits, dim=-1)
    with torch.no_grad():
        true_dist = torch.zeros_like(logits)
        true_dist.fill_(smoothing / (n_class - 1))
        true_dist.scatter_(1, target.unsqueeze(1), 1.0 - smoothing)
    return - (true_dist * log_probs).sum(dim=-1).mean()

def focal_loss(logits, target, gamma=2.0, alpha=0.25):
    ce = F.cross_entropy(logits, target, reduction='none')
    p_t = torch.exp(-ce)
    loss = alpha * ((1-p_t)**gamma) * ce
    return loss.mean()

def smooth_l1(pred, target, beta=1.0):
    diff = torch.abs(pred - target)
    loss = torch.where(diff < beta, 0.5 * diff**2 / beta, diff - 0.5*beta)
    return loss.mean()

def triplet_loss(anchor, positive, negative, margin=1.0):
    pos_dist = F.pairwise_distance(anchor, positive)
    neg_dist = F.pairwise_distance(anchor, negative)
    return F.relu(pos_dist - neg_dist + margin).mean()

def dice_loss(pred, target, eps=1e-6):
    pred = torch.sigmoid(pred)
    inter = (pred * target).sum(dim=(1,2,3))
    denom = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    loss = 1 - (2*inter + eps) / (denom + eps)
    return loss.mean()

def kl_divergence_logits(logits_p, logits_q):
    p = F.log_softmax(logits_p, dim=-1)
    q = F.softmax(logits_q, dim=-1)
    return F.kl_div(p, q, reduction='batchmean', log_target=False)

def bce_with_logits_loss(logits, targets):
    loss = torch.clamp(logits, min=0) - logits * targets + torch.log1p(torch.exp(-torch.abs(logits)))
    return loss.mean()

# 2.7 归一化 / 嵌入 / Patch Embedding
class LayerNormManual(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = ((x - mean) ** 2).mean(-1, keepdim=True)
        x_hat = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_hat + self.beta

class BatchNorm2dManual(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.eps = eps; self.momentum = momentum
        self.gamma = nn.Parameter(torch.ones(num_features))
        self.beta = nn.Parameter(torch.zeros(num_features))
        self.register_buffer('running_mean', torch.zeros(num_features))
        self.register_buffer('running_var', torch.ones(num_features))
        self.training=True
    def forward(self, x):
        if self.training:
            mean = x.mean(dim=[0,2,3])
            var = x.var(dim=[0,2,3], unbiased=False)
            self.running_mean = (1-self.momentum)*self.running_mean + self.momentum*mean.detach()
            self.running_var = (1-self.momentum)*self.running_var + self.momentum*var.detach()
            mean_, var_ = mean, var
        else:
            mean_, var_ = self.running_mean, self.running_var
        x_hat = (x - mean_[None,:,None,None]) / torch.sqrt(var_[None,:,None,None] + self.eps)
        return self.gamma[None,:,None,None] * x_hat + self.beta[None,:,None,None]

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
    def forward(self, idx):
        return self.embed(idx)  # (B, T, D)

class PatchEmbedding(nn.Module):
    def __init__(self, in_ch, embed_dim, patch_size=16):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        x = self.proj(x)  # (B, D, H/p, W/p)
        B, D, H, W = x.shape
        return x.flatten(2).transpose(1,2)  # (B, N, D)

print('Section 2 单元定义完成')

## Section 3: 计算机视觉核心

包含：IoU / GIoU / NMS / Soft-NMS / FPN / Anchor / ROIAlign / U-Net / Conv2d 手写 / ResNet / ViT Encoder

In [ ]:
# 3.1 IoU / GIoU（简洁实现，支持单框比较，教学用）
def bbox_iou(box1, box2, eps=1e-7):
    # box: (x1,y1,x2,y2) 单个框
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0])*(box1[3]-box1[1])
    area2 = (box2[2]-box2[0])*(box2[3]-box2[1])
    union = area1 + area2 - inter + eps
    return inter / union

def giou(box1, box2, eps=1e-7):
    iou = bbox_iou(box1, box2)
    cx1 = min(box1[0], box2[0])
    cy1 = min(box1[1], box2[1])
    cx2 = max(box1[2], box2[2])
    cy2 = max(box1[3], box2[3])
    c_area = (cx2-cx1)*(cy2-cy1) + eps
    # 重新计算 union
    inter_x1 = max(box1[0], box2[0])
    inter_y1 = max(box1[1], box2[1])
    inter_x2 = min(box1[2], box2[2])
    inter_y2 = min(box1[3], box2[3])
    inter = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    area1 = (box1[2]-box1[0])*(box1[3]-box1[1])
    area2 = (box2[2]-box2[0])*(box2[3]-box2[1])
    union = area1 + area2 - inter + eps
    return iou - (c_area - union) / c_area

print('IoU / GIoU 函数定义完成')

#############################################################################
# 3.2 NMS / Soft-NMS（简洁版）
def box_iou(boxes1, boxes2):
    # boxes: (N,4) & (M,4) -> (N,M) IoU
    N = boxes1.shape[0]
    M = boxes2.shape[0]
    ious = torch.zeros(N, M)
    for i in range(N):
        for j in range(M):
            ious[i,j] = bbox_iou(boxes1[i], boxes2[j])
    return ious

def nms(boxes, scores, iou_threshold=0.5):
    idxs = scores.argsort(descending=True)
    keep = []
    while idxs.numel() > 0:
        i = idxs[0].item()
        keep.append(i)
        if idxs.numel() == 1:
            break
        ious = box_iou(boxes[i].unsqueeze(0), boxes[idxs[1:]])[0]
        idxs = idxs[1:][ious <= iou_threshold]
    return keep

def soft_nms(boxes, scores, sigma=0.5, iou_thresh=0.3, method='linear'):
    boxes = boxes.clone(); scores = scores.clone()
    N = boxes.shape[0]
    for i in range(N):
        maxpos = i + torch.argmax(scores[i:]).item()
        boxes[i], boxes[maxpos] = boxes[maxpos].clone(), boxes[i].clone()
        scores[i], scores[maxpos] = scores[maxpos].clone(), scores[i].clone()
        if i == N-1: break
        ious = box_iou(boxes[i].unsqueeze(0), boxes[i+1:])[0]
        for j, iou in enumerate(ious):
            if method == 'linear':
                if iou > iou_thresh:
                    scores[i+1+j] *= (1 - iou)
            else:
                scores[i+1+j] *= torch.exp(- (iou**2) / sigma)
    keep = torch.where(scores > 0.001)[0].tolist()
    return keep

print('NMS / Soft-NMS 定义完成')

#############################################################################
# 3.3 FPN 简洁骨架
class SimpleFPN(nn.Module):
    def __init__(self, in_channels_list, out_channels=256):
        super().__init__()
        self.lateral_convs = nn.ModuleList([nn.Conv2d(c, out_channels, 1) for c in in_channels_list])
        self.output_convs = nn.ModuleList([nn.Conv2d(out_channels, out_channels, 3, padding=1) for _ in in_channels_list])
    def forward(self, features):
        last_inner = self.lateral_convs[-1](features[-1])
        results = [self.output_convs[-1](last_inner)]
        for i in range(len(features)-2, -1, -1):
            lateral = self.lateral_convs[i](features[i])
            inner_up = F.interpolate(last_inner, size=lateral.shape[-2:], mode='nearest')
            last_inner = lateral + inner_up
            results.insert(0, self.output_convs[i](last_inner))
        return results

print('SimpleFPN 定义完成')

#############################################################################
# 3.4 Anchor 生成（网格）
def generate_anchors(grid_size, scales, ratios, stride):
    H, W = grid_size
    centers_x = (torch.arange(W) + 0.5) * stride
    centers_y = (torch.arange(H) + 0.5) * stride
    gx, gy = torch.meshgrid(centers_x, centers_y, indexing='xy')
    centers = torch.stack([gx.flatten(), gy.flatten()], dim=1)
    anchors = []
    for s in scales:
        for r in ratios:
            w = s*math.sqrt(r); h = s/math.sqrt(r)
            boxes = torch.cat([centers - torch.tensor([w/2,h/2]), centers + torch.tensor([w/2,h/2])], dim=1)
            anchors.append(boxes)
    return torch.cat(anchors, dim=0)

print('Anchor 生成函数定义完成')

#############################################################################
# 3.5 ROI Align（教学版 bilinear 插值）
def bilinear_interpolate(im, x, y):
    # im: (C, H, W) x,y scalars
    x0 = int(torch.floor(x).item()); x1 = x0 + 1
    y0 = int(torch.floor(y).item()); y1 = y0 + 1
    x0c = max(0, min(x0, im.shape[2]-1)); x1c = max(0, min(x1, im.shape[2]-1))
    y0c = max(0, min(y0, im.shape[1]-1)); y1c = max(0, min(y1, im.shape[1]-1))
    wa = (x1 - x) * (y1 - y)
    wb = (x1 - x) * (y - y0)
    wc = (x - x0) * (y1 - y)
    wd = (x - x0) * (y - y0)
    Ia = im[:, y0c, x0c]; Ib = im[:, y1c, x0c]; Ic = im[:, y0c, x1c]; Id = im[:, y1c, x1c]
    return wa*Ia + wb*Ib + wc*Ic + wd*Id

def roi_align_single(feature, box, output_size=(7,7)):
    C,H,W = feature.shape
    x1,y1,x2,y2 = box
    w = max(x2-x1, 1e-6); h = max(y2-y1, 1e-6)
    out = torch.zeros(C, output_size[0], output_size[1])
    for i in range(output_size[0]):
        for j in range(output_size[1]):
            px = x1 + (j + 0.5) * w / output_size[1]
            py = y1 + (i + 0.5) * h / output_size[0]
            out[:,i,j] = bilinear_interpolate(feature, px, py)
    return out

print('ROI Align（教学版）定义完成')

#############################################################################
# 3.6 U-Net 简洁骨架
class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=1),
                                  nn.ReLU(),
                                  nn.Conv2d(out_ch, out_ch, 3, padding=1),
                                  nn.ReLU())
    def forward(self, x):
        return self.conv(x)

class SimpleUNet(nn.Module):
    def __init__(self, in_ch=3, base=32):
        super().__init__()
        self.enc1 = UNetBlock(in_ch, base)
        self.enc2 = UNetBlock(base, base*2)
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.dec1 = UNetBlock(base*3, base)
        self.final = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        u = self.up(e2)
        u = torch.cat([u, e1], dim=1)
        d = self.dec1(u)
        return self.final(d)

print('Simple U-Net 定义完成')

#############################################################################
# 3.7 Conv2d 手写版（极慢，教学用）
def conv2d_naive(x, weight, bias=None, stride=1, padding=0):
    # x: (C_in, H, W), weight: (C_out, C_in, kH, kW)
    C_in,H,W = x.shape
    C_out,_,kH,kW = weight.shape
    H_out = (H + 2*padding - kH) // stride + 1
    W_out = (W + 2*padding - kW) // stride + 1
    x_pad = F.pad(x.unsqueeze(0), (padding,padding,padding,padding)).squeeze(0)
    out = torch.zeros(C_out, H_out, W_out)
    for co in range(C_out):
        for i in range(H_out):
            for j in range(W_out):
                patch = x_pad[:, i*stride:i*stride+kH, j*stride:j*stride+kW]
                out[co,i,j] = (patch * weight[co]).sum()
    if bias is not None:
        out += bias.view(-1,1,1)
    return out

print('Conv2d 手写版定义完成')

#############################################################################
# 3.8 ResNet Residual Block（简洁）
class ResidualBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1), nn.BatchNorm2d(ch), nn.ReLU(),
            nn.Conv2d(ch, ch, 3, padding=1), nn.BatchNorm2d(ch)
        )
        self.relu = nn.ReLU()
    def forward(self, x):
        return self.relu(x + self.net(x))

print('ResNet block 定义完成')

#############################################################################
# 3.9 ViT Encoder（简洁）
class SimpleViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_ch=3, embed_dim=768, depth=6, num_heads=8):
        super().__init__()
        self.patch_emb = PatchEmbedding(in_ch, embed_dim, patch_size)
        num_patches = (img_size // patch_size)**2
        self.cls_token = nn.Parameter(torch.zeros(1,1,embed_dim))
        self.pos_emb = nn.Parameter(torch.zeros(1, num_patches+1, embed_dim))
        self.layers = nn.ModuleList([nn.TransformerEncoderLayer(embed_dim, nhead=num_heads) for _ in range(depth)])
    def forward(self, x):
        x = self.patch_emb(x)  # (B, N, D)
        B = x.shape[0]
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_emb
        for layer in self.layers:
            x = layer(x)
        return x[:,0]

print('Simple ViT 定义完成')

## Section 4: 强化学习（简洁实现）

包含：DQN (Replay Buffer + 目标网络)、A2C、SFT/RM/PPO/DPO 思路要点。

In [ ]:
# 4.1 DQN（经验回放 + 目标网络 + ε-greedy）
class ReplayBuffer:
    def __init__(self, size=10000):
        self.buf=[]; self.max=size
    def push(self, transition):
        if len(self.buf)>=self.max: self.buf.pop(0)
        self.buf.append(transition)
    def sample(self, batch):
        return random.sample(self.buf, batch)

class DQNAgent:
    def __init__(self, obs_dim, act_dim, hidden=128, lr=1e-3, gamma=0.99):
        self.q = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Linear(hidden, act_dim))
        self.target = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Linear(hidden, act_dim))
        self.target.load_state_dict(self.q.state_dict())
        self.optim = torch.optim.Adam(self.q.parameters(), lr=lr)
        self.gamma = gamma
    def act(self, obs, eps=0.1):
        if random.random() < eps:
            return random.randrange(self.q[-1].out_features)
        with torch.no_grad():
            return int(self.q(obs.unsqueeze(0)).argmax().item())
    def train_step(self, batch):
        obs, act, rew, next_obs, done = batch
        q_vals = self.q(obs).gather(1, act.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q = self.target(next_obs).max(1)[0]
            target = rew + self.gamma * next_q * (1-done)
        loss = F.mse_loss(q_vals, target)
        self.optim.zero_grad(); loss.backward(); self.optim.step()
        return loss.item()

print('DQNAgent 定义完成')

#############################################################################
# 4.2 A2C（简洁结构）
class A2CAgent(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=128):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(obs_dim, hidden), nn.ReLU())
        self.policy = nn.Linear(hidden, act_dim)
        self.value = nn.Linear(hidden, 1)
    def forward(self, x):
        h = self.shared(x)
        return F.softmax(self.policy(h), dim=-1), self.value(h).squeeze(-1)

print('A2C Agent 定义完成')

#############################################################################
# 4.3 RLHF 相关思路（SFT / RM / PPO / DPO 简要说明）
print('SFT: 使用 (prompt, response) 对对原语言模型做监督微调，直接用 CE loss')
print('RM: 奖励模型通常用 pairwise loss，例如 给定 (chosen, rejected) 最大化 P(chosen>rejected)')
print('PPO: 使用概率比值 r = pi_theta / pi_ref，对策略损失使用裁剪项以稳定训练')
print('DPO: 直接偏好优化，优化公式基于对比模型与参考模型的 log-prob 差值，避免训练奖励模型的复杂性')

## Section 5: Transformer 全架构 / 进阶模块

包含：Encoder-Decoder、Decoder 的 Masked Self-Attention & Cross-Attention、FFN / Gated FFN、Head 拆分/拼接、Transformer-XL（缓存）示意。

In [ ]:
# 5.1 Transformer Decoder Layer（含 self-attn + cross-attn）
class SimpleTransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)  # cross-attn
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model); self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None):
        tgt2,_ = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask)
        tgt = self.norm1(tgt + self.dropout(tgt2))
        tgt2,_ = self.multihead_attn(tgt, memory, memory, attn_mask=memory_mask)
        tgt = self.norm2(tgt + self.dropout(tgt2))
        tgt2 = self.linear2(F.relu(self.linear1(tgt)))
        tgt = self.norm3(tgt + self.dropout(tgt2))
        return tgt

print('Transformer Decoder 层定义完成')

#############################################################################
# 5.2 Gated FFN（如 GLU 风格门控 FFN）
class GatedFFN(nn.Module):
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim*2)
        self.w2 = nn.Linear(hidden_dim, dim)
    def forward(self, x):
        x_proj = self.w1(x)
        a, b = x_proj.chunk(2, dim=-1)
        return self.w2(F.gelu(a) * b)

print('Gated FFN 定义完成')

#############################################################################
# 5.3 Heads 拆分/合并函数（常考）
def split_heads(x, num_heads):
    B,T,D = x.shape
    return x.view(B, T, num_heads, D//num_heads).permute(0,2,1,3)

def combine_heads(x):
    B,H,T,hd = x.shape
    return x.permute(0,2,1,3).contiguous().view(B, T, H*hd)

print('Heads 拆分/合并函数定义完成')

#############################################################################
# 5.4 Transformer-XL 缓存示意（面试时画图 + 说明即可）
print('Transformer-XL: 将 previous segment 的 key/value 缓存并 concat 到新的 k/v 前面，以实现跨段依赖；使用相对位置编码。')

## Section 6: 长文本 / 高效训练 / 工程实现

包含：Sliding Window / Longformer / RoPE 插值 / LoRA / 生成策略 / 数值稳定性等。

In [ ]:
# 6.1 Sliding Window Attention（局部注意力）
def sliding_window_attention(q, k, v, window_size=128):
    B, T, D = q.shape
    out = torch.zeros_like(q)
    for i in range(T):
        l = max(0, i-window_size)
        r = min(T, i+window_size+1)
        scores = torch.einsum("bkd,bkd->bk", q[:,i:i+1,:].expand(-1,r-l,-1), k[:,l:r,:]) / math.sqrt(D)
        attn = torch.softmax(scores, dim=-1)
        out[:,i,:] = torch.einsum("bk,bkd->bd", attn, v[:,l:r,:])
    return out

print('Sliding window attention 定义完成')

#############################################################################
# 6.2 Longformer 思路（局部 + 全局 token）
def longformer_attention(q,k,v, global_idxs, window):
    B,T,D = q.shape
    local = sliding_window_attention(q,k,v, window_size=window)
    for idx in global_idxs:
        scores = torch.einsum("bkd,bkd->bk", q[:,idx:idx+1,:].expand(-1,T,-1), k) / math.sqrt(D)
        attn = torch.softmax(scores, dim=-1)
        local[:,idx,:] = torch.einsum("bk,bkd->bd", attn, v)
    return local

print('Longformer attention 思路函数定义完成')

#############################################################################
# 6.3 RoPE 插值 / 动态 RoPE 示例函数（示意）
def rope_interpolate(old_cos, old_sin, new_len):
    L_old = old_cos.shape[0]
    scale = torch.linspace(0, 1, new_len).unsqueeze(1)
    idx_old = (scale * (L_old-1)).long().squeeze(1)
    return old_cos[idx_old], old_sin[idx_old]

print('RoPE 插值示意函数定义完成')

#############################################################################
# 6.4 LoRA（低秩适配）简洁实现
class LoRALayer(nn.Module):
    def __init__(self, linear: nn.Linear, r=4, alpha=1.0):
        super().__init__()
        self.linear = linear
        dim_in = linear.in_features
        dim_out = linear.out_features
        self.r = r
        self.lora_A = nn.Parameter(torch.zeros(r, dim_in))
        self.lora_B = nn.Parameter(torch.zeros(dim_out, r))
        self.scaling = alpha / r
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
    def forward(self, x):
        return self.linear(x) + (x @ self.lora_A.t() @ self.lora_B.t()) * self.scaling

print('LoRA 层定义完成')

#############################################################################
# 6.5 生成策略示例（Greedy / Top-k / Top-p）
def greedy_decode(step_fn, input_ids, max_len):
    for _ in range(max_len):
        logits = step_fn(input_ids)
        next_token = logits[:, -1].argmax(dim=-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_token], dim=1)
    return input_ids

def top_k_top_p_sample(logits, top_k=0, top_p=0.0):
    probs = F.softmax(logits, dim=-1)
    if top_k > 0:
        topk_vals, topk_idx = torch.topk(probs, top_k)
        mask = torch.ones_like(probs, dtype=torch.bool)
        mask[topk_idx] = False
        probs = probs.masked_fill(mask, 0.0)
    if top_p > 0:
        sorted_probs, sorted_idx = torch.sort(probs, descending=True)
        cumsum = torch.cumsum(sorted_probs, dim=0)
        cutoff_idx = (cumsum > top_p).nonzero()
        if cutoff_idx.numel() > 0:
            cutoff = cutoff_idx[0].item()
            probs[sorted_idx[cutoff+1:]] = 0.0
    probs = probs / probs.sum()
    return torch.multinomial(probs, 1)

#############################################################################
# 6.6 数值稳定性：stable softmax 与 梯度裁剪
def stable_softmax(x, dim=-1):
    x = x - x.max(dim=dim, keepdim=True)[0]
    e = torch.exp(x)
    return e / e.sum(dim=dim, keepdim=True)

def clip_grad_norm_(parameters, max_norm, eps=1e-6):
    total_norm = 0.0
    for p in parameters:
        if p.grad is None: continue
        param_norm = p.grad.data.norm(2)
        total_norm += param_norm.item() ** 2
    total_norm = total_norm ** 0.5
    clip_coef = max_norm / (total_norm + eps)
    if clip_coef < 1:
        for p in parameters:
            if p.grad is None: continue
            p.grad.data.mul_(clip_coef)
    return total_norm

print('Section 6 单元定义完成')

## 结束语

本 notebook 提供了面试常见手写题点的教学级实现与注释。很多实现为简化版以突出核心思想，面试时建议：
- 先写公式并说明复杂度（时间/空间）、数值稳定性问题
- 写出简洁可运行的代码（如本 notebook），并说明工程优化方向（FlashAttention kernel、并行/分布式、16-bit/8-bit 等）
- 如需我可把此 .ipynb 文件打包下载、或把某一模块（如 FlashAttention 的精确数值稳定分块实现、LoRA+4-bit 的实操代码）进一步工程化并给出测试基准。

你希望我现在把这个 notebook 打包为下载链接（附件），还是先把某些模块做更工程化的实现？